### Define the pydantic model (Schema + validation)

In [1]:
from pydantic import BaseModel,Field
from typing import Annotated

class Person(BaseModel):
    name:Annotated[str,Field(...,description="Name of the person")]
    age:Annotated[int,Field(...,gt=18,description="Age of the person")] #Validation:age >18
    city:Annotated[str,Field(...,description="Name of the city the person belongs to")]

### 2.Initialize the Model and the parser

In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("Hugging_face_api_token")

# Create LLM endpoint
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)

# Wrap with chat interface
model = ChatHuggingFace(llm=llm)

In [5]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser=PydanticOutputParser(pydantic_object=Person)

### 3.Create Prompt Template with Format Instructions

In [7]:
from langchain_core.prompts import PromptTemplate

template=PromptTemplate(
    template="""Generate the name,age and city of a fictional {place} person \n {format_instruction}""",
    input_variables=["place"],
    partial_variables={
        "format_instruction":pydantic_parser.get_format_instructions()
    }
)

### 4.Create and Invoke Chains

In [8]:
chain=template | model | pydantic_parser
result=chain.invoke({
    "place":"Nepalese"
})

### 5.Output

In [ ]:
print(f"Output: {result} ")
print(type(result)) #pydantic object

Output: name='Sagar Parajuli' age=25 city='Kathmandu' 
<class '__main__.Person'>


### convert it to dict  for taking keys

In [11]:
result_dict=result.model_dump()
type(result_dict)

dict

In [12]:
result_dict

{'name': 'Sagar Parajuli', 'age': 25, 'city': 'Kathmandu'}

In [13]:
result_dict.keys()

dict_keys(['name', 'age', 'city'])